CloudFront is AWS's global Content Delivery Network (CDN) — it caches content at edge locations worldwide to reduce latency for end users and offload traffic from your origin servers. Global Accelerator uses the same AWS backbone to accelerate non-cacheable workloads like APIs, gaming, and VoIP. Together they are AWS's two answers to the problem of serving users globally with low latency.

## CloudFront Fundamentals

CloudFront sits between your users and your origin servers. When a user requests content:

```text
User (Sydney)
  │
  ▼  DNS resolves to nearest edge location
CloudFront Edge (Sydney)
  │
  ├── Cache HIT  → return cached content immediately (low latency)
  │
  └── Cache MISS → forward to origin, cache response, return to user
                    Origin: S3 / ALB / EC2 / HTTP server
```

### Key numbers
- **400+ edge locations** (Points of Presence) globally
- **13 regional edge caches** — larger mid-tier caches between edge and origin
- Content served over AWS backbone (not public internet) from edge to origin

### Benefits
- Reduced latency — content served from the edge nearest the user
- Reduced origin load — cache absorbs repeated requests
- DDoS protection — AWS Shield Standard included automatically
- HTTPS termination at the edge
- Global reach without deploying infrastructure worldwide

## Origins

An **origin** is where CloudFront fetches content when it has a cache miss.

| Origin type | Notes |
|---|---|
| **S3 bucket** | Most common for static assets; use Origin Access Control (OAC) to keep bucket private |
| **S3 static website** | Use HTTP endpoint — OAC does not apply; treat as custom origin |
| **Application Load Balancer** | For dynamic content and APIs; ALB must be public |
| **EC2 instance** | Must have a public IP or be reachable |
| **Any HTTP server** | On-premises, another cloud, any public HTTPS endpoint |

### Origin Access Control (OAC)
OAC is the recommended way to restrict S3 bucket access so **only CloudFront** can read from it — the bucket itself stays private, no public access needed.

```text
User → CloudFront (OAC signs requests) → Private S3 bucket
                                               ↑
               Bucket policy: allow only CloudFront service principal
```

OAC replaced the older **Origin Access Identity (OAI)**. OAC supports:
- S3 SSE-KMS encrypted buckets
- All S3 regions including newer regions
- Dynamic requests (PUT, DELETE)

### Origin Groups (failover)
An **Origin Group** has a primary and a secondary origin. If the primary returns a 5xx error or times out, CloudFront automatically retries the secondary. Used for origin-level failover — e.g. primary S3 bucket in us-east-1, secondary in eu-west-1.

## Cache Behaviours

A **cache behaviour** maps a URL path pattern to an origin and controls caching settings for that path.

```text
Distribution: d1234.cloudfront.net
├── /api/*         → ALB origin      (TTL: 0 — no caching for dynamic)
├── /images/*      → S3 origin       (TTL: 86400 — cache 24h)
└── /* (default)   → S3 origin       (TTL: 3600 — cache 1h)
```

Each behaviour can configure:
- Which origin to use
- **Viewer protocol policy**: HTTP only, HTTPS only, redirect HTTP→HTTPS
- **Allowed HTTP methods**: GET/HEAD, GET/HEAD/OPTIONS, or all methods
- **Cache policy**: what to include in the cache key (headers, cookies, query strings)
- **TTL**: min, max, and default TTL
- **Lambda@Edge or CloudFront Functions** to attach

### TTL and cache control
- CloudFront respects `Cache-Control` and `Expires` headers from the origin
- **Default TTL**: 24 hours if origin sends no cache headers
- **Minimum TTL**: 0 (forces CloudFront to always check with origin)
- You can **invalidate** cached objects manually — costs money per path; use versioned filenames (e.g. `app.v2.js`) instead

### Cache key
By default the cache key is just the URL path. You can add:
- **Query strings** — `?version=2` creates a separate cache entry
- **Headers** — e.g. `Accept-Language` to serve localised content
- **Cookies** — for user-specific content

> Adding more items to the cache key improves cache correctness but reduces the **cache hit ratio** — more unique cache keys = more origin fetches.

## CloudFront Security

### HTTPS
- CloudFront provides a default certificate for `*.cloudfront.net`
- For custom domains: attach an **ACM (AWS Certificate Manager)** certificate — must be in **us-east-1** regardless of distribution's regions
- **Viewer protocol**: enforce HTTPS between user and CloudFront
- **Origin protocol**: enforce HTTPS between CloudFront and origin

### Geo restriction
Block or allow requests based on the user's country (determined by IP geolocation):
- **Allowlist**: only listed countries can access
- **Blocklist**: listed countries are blocked
- Returns 403 to blocked requests

### Signed URLs and Signed Cookies
Used to grant **temporary, controlled access** to private content — e.g. paid video streams, premium downloads.

| | Signed URL | Signed Cookie |
|---|---|---|
| Scope | One specific file | Multiple files / entire distribution |
| Use case | Single premium download | Subscription content (many files) |
| How | URL contains expiry + signature | Cookie contains expiry + signature |

### AWS WAF integration
Attach a **WAF (Web Application Firewall)** web ACL to a CloudFront distribution to:
- Block SQL injection and XSS
- Rate limit requests
- Block specific IPs or countries (in addition to geo restriction)
- Allow/block based on request headers or body

### Field-Level Encryption
Encrypt specific POST fields (e.g. credit card numbers) at the edge using a public key — only your backend with the private key can decrypt. The field stays encrypted even as it passes through CloudFront and your application layers.

## Edge Functions: Lambda@Edge vs CloudFront Functions

Run code at CloudFront edge locations to customise HTTP requests and responses.

| | CloudFront Functions | Lambda@Edge |
|---|---|---|
| **Runtime** | JavaScript (ES5) | Node.js, Python |
| **Trigger** | Viewer request/response only | Viewer + origin request/response |
| **Max execution time** | 1 ms | 5 sec (viewer), 30 sec (origin) |
| **Max memory** | 2 MB | 128 MB – 10 GB |
| **Network access** | No | Yes |
| **Cost** | ~6× cheaper | More expensive |
| **Scale** | Millions of req/sec | Thousands of req/sec |

### CloudFront Functions use cases
- URL rewrites and redirects
- HTTP header manipulation (add security headers)
- Cache key normalisation (lowercase query strings)
- Simple JWT validation

### Lambda@Edge use cases
- Dynamic content generation
- A/B testing (modify origin based on cookie)
- User authentication against an external IdP
- Fetching data from DynamoDB to personalise responses
- Origin selection based on request properties

### Request/response lifecycle
```text
User → [Viewer Request] → CloudFront Cache
                               │ cache miss
                               ▼
                        [Origin Request] → Origin
                                               │
                        [Origin Response] ←────┘
                               │
                        [Viewer Response] → User
```
CloudFront Functions only run at viewer request/response. Lambda@Edge can run at all four points.

## Price Classes and Costs

CloudFront pricing varies by edge location region. **Price Classes** let you limit which edge locations are used:

| Price Class | Edge locations included |
|---|---|
| **Price Class All** | All locations worldwide (lowest latency, highest cost) |
| **Price Class 200** | All except South America, Australia, New Zealand |
| **Price Class 100** | North America and Europe only (cheapest) |

Users outside your price class region are served from the nearest included edge location — slightly higher latency but lower cost.

### Free tier
- 1 TB data transfer out per month
- 10 million HTTP/HTTPS requests per month
- (12 months free tier for new accounts)

## CloudFront vs S3 Cross-Region Replication

These are often confused — they solve different problems:

| | CloudFront | S3 Cross-Region Replication |
|---|---|---|
| **What it does** | Caches content at edge locations | Replicates objects to another S3 bucket |
| **Latency** | Low (edge near user) | Low (data pre-positioned in target region) |
| **Content freshness** | TTL-based cache; stale until TTL expires | Near real-time replication |
| **File size** | Any | Any |
| **Read/write** | Read-only caching | Full S3 read/write in target bucket |
| **Use case** | Static content globally, high cache hit ratio | DR, specific region low latency, compliance |
| **Coverage** | 400+ locations automatically | Specific regions you choose |

> **Exam tip:** CloudFront = cache everywhere automatically. S3 Replication = copy to specific regions, always fresh.

## AWS Global Accelerator

Global Accelerator improves the performance of global applications by routing traffic through the **AWS global backbone network** instead of the public internet.

### How it works
```text
User (Tokyo)
  │  short public internet hop
  ▼
AWS Edge Location (Tokyo)  ← enters AWS backbone here
  │  travels over AWS private backbone (low latency, consistent)
  ▼
Your Application (us-east-1 ALB)
```

Without Global Accelerator, the entire path from Tokyo to us-east-1 goes over the public internet — many unpredictable hops, variable latency.

### Anycast IP addresses
Global Accelerator provides **2 static Anycast IPv4 addresses** for your application. Anycast means the same IP is announced from multiple AWS edge locations — the user's traffic is automatically routed to the nearest one.

Benefits of static IPs:
- Clients hardcode IPs (no DNS TTL delays for failover)
- Works for applications that cannot use DNS-based routing
- Whitelisting is simple — just 2 IPs

### Endpoints
Global Accelerator can route to:
- Application Load Balancers
- Network Load Balancers
- EC2 instances (with Elastic IPs)
- Elastic IPs

Endpoints can be in **multiple regions** — Global Accelerator routes to the healthiest, lowest-latency endpoint automatically.

### Traffic controls
- **Traffic dials** — shift a percentage of traffic away from an endpoint group (for deployments or maintenance)
- **Endpoint weights** — distribute traffic within an endpoint group
- **Health checks** — built-in, automatic failover when an endpoint becomes unhealthy

### Global Accelerator vs CloudFront

| | CloudFront | Global Accelerator |
|---|---|---|
| **Layer** | Application (HTTP/HTTPS) | Network (TCP/UDP) |
| **Caching** | Yes — content cached at edge | No — no caching, just routing |
| **Protocols** | HTTP, HTTPS, WebSocket | TCP, UDP |
| **IP addresses** | Changes (DNS-based) | 2 static Anycast IPs |
| **Use case** | Static/cacheable content, websites | APIs, gaming, VoIP, IoT, non-HTTP |
| **Failover speed** | DNS TTL (minutes) | Near-instant (network level) |

> **Exam tip:** If the question mentions **static IP**, **non-HTTP protocol**, **gaming**, **IoT**, **VoIP**, or **instant failover** → Global Accelerator. If it mentions **caching**, **CDN**, **static website**, or **media delivery** → CloudFront.

## Working with CloudFront via boto3

In [ ]:
import boto3
import uuid

cf = boto3.client('cloudfront')
ga = boto3.client('globalaccelerator', region_name='us-east-1')  # GA is global but API is us-east-1

In [ ]:
# Step 1: Create an Origin Access Control for S3
oac = cf.create_origin_access_control(
    OriginAccessControlConfig={
        'Name': 'my-s3-oac',
        'Description': 'OAC for private S3 bucket',
        'SigningProtocol': 'sigv4',
        'SigningBehavior': 'always',
        'OriginAccessControlOriginType': 's3'
    }
)
oac_id = oac['OriginAccessControl']['Id']
print(f"OAC ID: {oac_id}")

In [ ]:
# Step 2: Create a CloudFront distribution with S3 origin + OAC
dist = cf.create_distribution(
    DistributionConfig={
        'CallerReference': str(uuid.uuid4()),
        'Comment': 'Static website distribution',
        'Enabled': True,
        'DefaultRootObject': 'index.html',
        'Origins': {
            'Quantity': 1,
            'Items': [{
                'Id': 's3-origin',
                'DomainName': 'my-bucket.s3.us-east-1.amazonaws.com',
                'S3OriginConfig': {'OriginAccessIdentity': ''},  # empty when using OAC
                'OriginAccessControlId': oac_id
            }]
        },
        'DefaultCacheBehavior': {
            'TargetOriginId': 's3-origin',
            'ViewerProtocolPolicy': 'redirect-to-https',
            'AllowedMethods': {'Quantity': 2, 'Items': ['GET', 'HEAD'],
                               'CachedMethods': {'Quantity': 2, 'Items': ['GET', 'HEAD']}},
            'CachePolicyId': '658327ea-f89d-4fab-a63d-7e88639e58f6',  # CachingOptimized managed policy
            'Compress': True
        },
        'PriceClass': 'PriceClass_100',  # North America + Europe only
        'HttpVersion': 'http2and3',
        'ViewerCertificate': {
            'CloudFrontDefaultCertificate': True  # use *.cloudfront.net cert
        }
    }
)
dist_id = dist['Distribution']['Id']
domain = dist['Distribution']['DomainName']
print(f"Distribution: {dist_id}")
print(f"Domain: {domain}")

In [ ]:
# Invalidate cached objects (e.g. after a deployment)
# Prefer versioned filenames (app.v2.js) to avoid invalidation costs
cf.create_invalidation(
    DistributionId=dist_id,
    InvalidationBatch={
        'Paths': {
            'Quantity': 2,
            'Items': ['/index.html', '/static/*']  # wildcard supported
        },
        'CallerReference': str(uuid.uuid4())
    }
)
print("Cache invalidation submitted")

In [ ]:
# Generate a signed URL for a private object (using canned policy)
from datetime import datetime, timedelta, timezone
from botocore.signers import CloudFrontSigner
import rsa

# In production: load private key from Secrets Manager or KMS
# cf_private_key = ... (RSA 2048-bit PEM)
# cf_key_pair_id = 'KXXXXXXXXXXXXX'

# def rsa_signer(message):
#     return rsa.sign(message, rsa.PrivateKey.load_pkcs1(cf_private_key), 'SHA-1')

# signer = CloudFrontSigner(cf_key_pair_id, rsa_signer)
# signed_url = signer.generate_presigned_url(
#     url='https://d1234.cloudfront.net/premium/video.mp4',
#     date_less_than=datetime.now(timezone.utc) + timedelta(hours=1)
# )
# print(signed_url)
print("Signed URL generation requires a CloudFront key pair — see comments above")

In [ ]:
# Create a Global Accelerator
accelerator = ga.create_accelerator(
    Name='my-global-app',
    IpAddressType='IPV4',
    Enabled=True,
    Tags=[{'Key': 'Environment', 'Value': 'production'}]
)
acc_arn = accelerator['Accelerator']['AcceleratorArn']
static_ips = [ip['IpAddress'] for ip in accelerator['Accelerator']['IpSets'][0]['IpAddresses']]
print(f"Accelerator: {acc_arn}")
print(f"Static IPs: {static_ips}")

# Add a listener (TCP port 443)
listener = ga.create_listener(
    AcceleratorArn=acc_arn,
    Protocol='TCP',
    PortRanges=[{'FromPort': 443, 'ToPort': 443}],
    ClientAffinity='SOURCE_IP'  # same client always goes to same endpoint
)
listener_arn = listener['Listener']['ListenerArn']

# Add endpoint groups per region
ga.create_endpoint_group(
    ListenerArn=listener_arn,
    EndpointGroupRegion='us-east-1',
    TrafficDialPercentage=100,
    EndpointConfigurations=[
        {
            'EndpointId': 'arn:aws:elasticloadbalancing:us-east-1:123456789012:loadbalancer/app/my-alb/abc123',
            'Weight': 100,
            'ClientIPPreservationEnabled': True
        }
    ]
)
print("Global Accelerator configured with us-east-1 ALB endpoint")

## Summary

| Concept | Key Takeaway |
|---|---|
| CloudFront | CDN — caches content at 400+ edge locations; reduces latency and origin load |
| OAC | Keeps S3 bucket private; only CloudFront can read; replaces OAI |
| Origin Group | Primary + secondary origin for automatic CloudFront-level failover |
| Cache behaviour | Path-pattern routing to different origins with different TTLs |
| TTL | Respect Cache-Control from origin; invalidation costs money — prefer versioned filenames |
| ACM certificate | Must be in us-east-1 for CloudFront custom domains |
| Geo restriction | Allow/block countries at CloudFront level |
| Signed URL | Temporary access to a single private file |
| Signed Cookie | Temporary access to multiple private files |
| CloudFront Functions | Viewer request/response; 1ms; URL rewrites, header manipulation |
| Lambda@Edge | All 4 trigger points; 5–30s; auth, personalisation, origin selection |
| Price Class 100 | North America + Europe only — cheapest |
| CloudFront vs S3 replication | CF = cached everywhere; S3 replication = specific regions, always fresh |
| Global Accelerator | 2 static Anycast IPs; AWS backbone routing; TCP/UDP; no caching |
| GA vs CloudFront | GA = non-HTTP, static IPs, instant failover; CF = HTTP caching, CDN |
| GA traffic dial | Gradually shift traffic away from an endpoint group for deployments |